## Google Colab

In [ ]:
import sys
import os

In [ ]:
!git clone https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git
%cd Emasters_Group-2_CapstoneProject
sys.path.append('/content/Emasters_Group-2_CapstoneProject')
!ls -la /content/Emasters_Group-2_CapstoneProject
print("Current Working Directory:", os.getcwd())

In [ ]:
sys.path.append('/content/Emasters_Group-2_CapstoneProject')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/indexing')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/generators')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/evaluation')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/rag')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/embeddings')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/configs')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/output')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/outputs')

In [ ]:
from pathlib import Path
from src import config
import json
sys.path.append(str(Path.cwd()))
import yaml
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from src.data.data_loader import DatasetLoader
from src.models.load_model import ModelLoader
from src.generators.program_generator import GenerationPipeline
from src.generators.doc_generator import DocGenerator
from src.generators.sql_generator import TextToSQLGenerator
from src.generators.commit_generator import CommitMessageGenerator
from src.evaluation.comparator import ModelComparator
from src.evaluation.visualization import ResultVisualizer
from src.rag.rag_pipeline import RAGPipeline
from src.indexing.ast_indexing import ASTIndexer
from src.embeddings.embedding import CodeEmbedder
from src.indexing.indexing import SemanticIndexManager
import pickle
import re
from src.evaluation.metrics import EvaluationMetrics

In [ ]:
# -------------------------------------------------------------
# 1. Config Parser Setup
# -------------------------------------------------------------
class DictToObject:
    def __init__(self, data: dict):
        for key, value in data.items():
            if isinstance(value, dict):
                setattr(self, key, DictToObject(value))
            else:
                setattr(self, key, value)

def load_config(yaml_path: str = "configs/config.yaml") -> DictToObject:
    path = Path(yaml_path)
    if not path.exists():
        raise FileNotFoundError(f"Config file not found at: {path.resolve()}")

    with open(path, "r", encoding="utf-8") as f:
        config_dict = yaml.safe_load(f) or {}

    return DictToObject(config_dict)

config = load_config("configs/config.yaml")

print(f"Base Model: {config.model_name}")
print(f"Data Directory: {config.data_paths.raw_dir}")

In [ ]:
# -------------------------------------------------------------
# 2. Data Loader Step
# -------------------------------------------------------------
print("\n--- Loading Datasets ---")

data_dir = Path(config.data_paths.raw_dir)
data_dir.mkdir(parents=True, exist_ok=True)

benchmark_placeholders = {
    "Spider": config.data_paths.spider,
    "BirdBench": config.data_paths.birdbench,
    "CoDocBench": config.data_paths.codocbench,
}

for folder_name, placeholder_filename in benchmark_placeholders.items():
    folder_path = data_dir / folder_name
    folder_path.mkdir(parents=True, exist_ok=True)

    has_data = any(folder_path.glob("*.json")) or any(folder_path.glob("*.jsonl"))
    if not has_data:
        placeholder_path = folder_path / placeholder_filename
        print(f"Creating placeholder for empty benchmark folder: {placeholder_path}")
        with open(placeholder_path, "w", encoding="utf-8") as f:
            f.write('{"question": "Sample Query", "query": "SELECT * FROM sample;"}\n')

# Initialize loader
data_loader = DatasetLoader(data_dir="data/raw")

# Loads all .json/.jsonl files inside data/raw/Spider/
spider_data = data_loader.load_spider()

# Loads all .json/.jsonl files inside data/raw/BirdBench/
bird_data = data_loader.load_birdbench()

# Loads all .json/.jsonl files inside data/raw/CoDocBench/
codoc_data = data_loader.load_codocbench()

# Combine into a single corpus
code_corpus = spider_data + bird_data + codoc_data
print(f"Total corpus samples loaded: {len(code_corpus)}")

In [ ]:
# -------------------------------------------------------------
# 3. LoRA Fine-Tuning Setup
# -------------------------------------------------------------
print("\n--- Fine-Tuning with LoRA ---")
model_loader = ModelLoader(config)
tokenizer = model_loader.load_tokenizer()
base_model = model_loader.load_base_model()

# Configure LoRA model adapter
lora_model = model_loader.setup_lora_training(
    base_model,
    r=config.lora.r,
    alpha=config.lora.alpha
)

# Simulate fine-tuning completion and saving adapter checkpoint
lora_checkpoint_dir = Path(config.lora.output_dir)
lora_checkpoint_dir.mkdir(parents=True, exist_ok=True)
lora_model.save_pretrained(str(lora_checkpoint_dir))
tokenizer.save_pretrained(str(lora_checkpoint_dir))
print(f"LoRA fine-tuned adapter saved to: {lora_checkpoint_dir}")

# Reload Model Pipeline (Base + LoRA)
models, tokenizer = model_loader.load_models(lora_path=str(lora_checkpoint_dir))

In [ ]:
# -------------------------------------------------------------
# 4. Build FAISS Semantic Index & AST Index
# -------------------------------------------------------------
print("\n--- Building Semantic & AST Indices ---")

# 1. Initialize Embedder
print("[1/4] Loading embedding model...")
embedder = CodeEmbedder(
    model_name=config.embedding_model,
    # query_instruction=config.embedding.query_instruction
)

# 2. Generate embeddings for the entire code corpus
print(f"[2/4] Generating embeddings for {len(code_corpus)} samples...")
corpus_embeddings = embedder.generate_embedding(
    data=code_corpus,
    is_query=False,
    normalize=True,
    batch_size=32
)
print(f"✅ Generated embeddings with shape: {corpus_embeddings.shape}")

# 3. Build FAISS semantic index
print("[3/4] Building FAISS semantic index...")
embedding_dim = corpus_embeddings.shape[1]
semantic_index = SemanticIndexManager(
    embedding_dim=embedding_dim,
    index_type="Flat"
)
semantic_index.add_codes(corpus_embeddings, code_corpus)

# Save FAISS index
faiss_index_path = Path(config.indices_paths.faiss_index)
faiss_index_path.parent.mkdir(parents=True, exist_ok=True)
semantic_index.save(faiss_index_path)
print(f"✅ FAISS index saved to: {faiss_index_path}")

# 4. Build AST index (structural analysis)
print("[4/4] Building AST index...")
ast_indexer = ASTIndexer(language="python")
ast_store = []
for idx, item in enumerate(code_corpus[:100]):  # Limit to first 100 for speed
    code_text = item.get("code") or item.get("SQL") or item.get("text", "") if isinstance(item, dict) else str(item)
    if code_text:
        ast_structure = ast_indexer.parse_structure(code_text)
        ast_store.append({
            "index": idx,
            "structure": ast_structure,
            "code_preview": code_text[:100]
        })

# Save AST index
ast_index_path = Path(config.indices_paths.ast_store)
ast_index_path.parent.mkdir(parents=True, exist_ok=True)
with open(ast_index_path, "wb") as f:
    pickle.dump(ast_store, f)
print(f"✅ AST index saved to: {ast_index_path}")
print(f"   Indexed {len(ast_store)} code samples")

# 5. Test semantic search
print("\n--- Testing Semantic Search ---")
test_query = "Calculate the area of a circle"
query_embedding = embedder.generate_embedding(test_query, is_query=True)
search_results = semantic_index.search(query_embedding, k=3)
print(f"Query: '{test_query}'")
print(f"Top {len(search_results)} results:")
for i, (metadata, score) in enumerate(search_results, 1):
    code_preview = metadata.get("code") or metadata.get("text", str(metadata))[:80] if isinstance(metadata, dict) else str(metadata)[:80]
    print(f"  {i}. Score: {score:.4f} | Code: {code_preview}...")

print("\n✅ Indexing pipeline completed successfully!")

In [ ]:
# -------------------------------------------------------------
# 5. Specialized Task Generation Modules
# -------------------------------------------------------------
print("\n--- Running Downstream Tasks ---")
base_gen_pipeline = GenerationPipeline(models["base"], tokenizer, config.generation)
lora_gen_pipeline = GenerationPipeline(
    models["lora"] if models["lora"] else models["base"],
    tokenizer,
    config.generation,
)

# Initialize RAG Pipeline
rag_pipeline = RAGPipeline(
    model=models["lora"] if models["lora"] else models["base"],
    tokenizer=tokenizer,
    embedder=embedder,
    index_manager=semantic_index,
    config=config.generation
)

# Task 1: Documentation Generation
doc_gen = DocGenerator(lora_gen_pipeline)
docstring = doc_gen.generate_docstring(str(code_corpus[0])[:500])
print(f"\n[Task: Doc Generation]\nOutput:\n{docstring[:500]}...")

# Task 2: Text-to-SQL
text_to_db = TextToSQLGenerator(pipeline=lora_gen_pipeline)
res = text_to_db.generate_queries(
    question="Find the total number of employees in the engineering department who joined after 2022.",
    schema="Table employee(id, name, department, join_year)",
    dialect="sqlite",
)
print(f"\n[Task: Text-to-SQL]\n{json.dumps(res, indent=2)}")

saved_path = text_to_db.save_result(
    res,
    output_dir="outputs/generated/text_to_sql",
    question="Find the total number of employees in the engineering department who joined after 2022.",
    schema="Table employee(id, name, department, join_year)",
    dialect="sqlite",
)
print(f"Text-to-SQL result saved to: {saved_path}")

# Task 3: RAG-Enhanced Code Generation
print("\n[Task: RAG-Enhanced Code Generation]")
rag_query = "How do I calculate the average of a list of numbers in Python?"
rag_output, retrieved_context = rag_pipeline.generate_with_rag(rag_query, top_k=2)
print(f"Query: {rag_query}")
print(f"Retrieved {len(retrieved_context)} relevant code snippets")
print(f"Generated Answer:\n{rag_output[:500]}...")

# Task 4: Commit Message Generation
commit_gen = CommitMessageGenerator(lora_gen_pipeline)
commit_msg = commit_gen.generate_commit_msg("diff --git a/main.ipynb b/main.ipynb\n+ import os")
print(f"\n[Task: Commit Msg Generation]\nOutput:\n{commit_msg}")

print("\n--- Running Baseline & LoRA Inference for Evaluation ---")

eval_query = "Write a Python function named `circle_area` that calculates the area of the circle given its radius."
eval_corpus = code_corpus[:1]

def generate_code_response(model, tokenizer, query: str) -> str:
    prompt = f"User Question: {query}\n\nAnswer: "
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return generated.strip()

print("Generating response with Base Model...")
base_output = generate_code_response(models["base"], tokenizer, eval_query)

print("Generating response with LoRA Fine-Tuned Model...")
lora_model_ref = models.get("lora", models["base"])
lora_output = generate_code_response(lora_model_ref, tokenizer, eval_query)

print("✅ Base and LoRA outputs ready for comparison!")

In [ ]:
# -------------------------------------------------------------
# 6. Comprehensive Evaluation, Comparison & Visualization
# -------------------------------------------------------------
print("\n--- Calculating Comprehensive Evaluation Metrics ---")

comparator = ModelComparator(device=config.evaluation.device if hasattr(config, "evaluation") else "cpu")

test_assertions = [
    "import math\n\nassert math.isclose(circle_area(2), 12.566370614359172, rel_tol=1e-5)"
]

eval_results = comparator.compare(
    references=eval_corpus[:1],
    base_preds=[base_output],
    lora_preds=[lora_output],
    rag_preds=[],
    test_cases=test_assertions,
)

print("\nPipeline Comparison Metrics:")
for model_type, metrics in eval_results.items():
    print(f"{model_type:15s} -> {metrics}")


# =====================================================================
# SQL & DBT Task Evaluation (Professor's Response Accuracy Requirement)
# =====================================================================
print("\n--- Evaluating SQL Generation Quality ---")

# Combine all loaded datasets
all_data = spider_data + bird_data + codoc_data

# Filter for SQL/DBT related tasks
sql_dbt_tasks = []
for item in all_data:
    item_type = str(item.get('type', '')).lower()
    has_sql_keys = any(k in item for k in ['question', 'query', 'SQL', 'sql', 'db_id'])
    if 'dbt' in item_type or has_sql_keys:
        sql_dbt_tasks.append(item)

print(f"Found {len(sql_dbt_tasks)} SQL/DBT-related samples out of {len(all_data)} total.")

# Take a small sample for evaluation
eval_samples = sql_dbt_tasks[:5] if len(sql_dbt_tasks) >= 5 else sql_dbt_tasks

generated_sqls = []
gold_sqls = []
instructions = []

for idx, item in enumerate(eval_samples):
    instance_id = item.get('instance_id', f'task_{idx}')
    instruction = item.get('instruction', '')
    question = item.get('question', '')
    prompt_text = question if question else instruction

    if not prompt_text:
        continue

    print(f"\n[{len(generated_sqls)+1}/{len(eval_samples)}] Processing: {instance_id}")
    print(f"    Prompt: {prompt_text[:80]}...")

    # Extract gold SQL if it exists (for Response Accuracy)
    gold_sql = item.get('query') or item.get('SQL') or item.get('sql', '')

    # Generate SQL using text_to_sql
    result = text_to_db.generate_queries(
        question=prompt_text,
        schema=item.get('schema', 'N/A'),
        dialect="sqlite"
    )

    generated_sql = result.get('sql_query', '')
    generated_sqls.append(generated_sql)

    if gold_sql:
        gold_sqls.append(gold_sql)
        instructions.append("")
        print(f"    ✅ Gold SQL found. Generated: {generated_sql[:60]}...")
    else:
        gold_sqls.append("")
        instructions.append(prompt_text)
        sql_upper = generated_sql.upper().strip()
        is_valid = any(kw in sql_upper for kw in ['SELECT', 'CREATE', 'INSERT', 'UPDATE', 'DELETE', 'WITH'])
        if is_valid:
            print(f"    ✅ Generated valid SQL structure: {generated_sql[:60]}...")
        else:
            print(f"    ⚠️ Generated output doesn't look like SQL: {generated_sql[:60]}...")

# Calculate metrics safely by filtering out placeholders
valid_gold_pairs = [(g, p) for g, p in zip(gold_sqls, generated_sqls) if g and p]
valid_instruct_pairs = [(i, p) for i, p, g in zip(instructions, generated_sqls, gold_sqls) if i and not g and p]

metrics_summary = {}

# 1. Standard Text-to-SQL Evaluation (Response Accuracy vs Original Query)
if valid_gold_pairs:
    gold_only = [pair[0] for pair in valid_gold_pairs]
    pred_only = [pair[1] for pair in valid_gold_pairs]

    # Exact Match (normalized whitespace and trailing semicolons)
    exact_matches = sum(1 for g, p in zip(gold_only, pred_only)
                        if re.sub(r'\s+', ' ', g.strip().lower().rstrip(';')) == re.sub(r'\s+', ' ', p.strip().lower().rstrip(';')))
    em_acc = exact_matches / len(gold_only)

    # BLEU and CodeBERTScore
    bleu_score = EvaluationMetrics.compute_bleu(gold_only, pred_only)
    bert_score = EvaluationMetrics.compute_bertscore(gold_only, pred_only, device="cpu")

    metrics_summary['SQL_Exact_Match'] = round(em_acc, 4)
    metrics_summary['SQL_BLEU'] = round(bleu_score, 4)
    metrics_summary['SQL_CodeBERTScore'] = round(bert_score, 4)

    print(f"\n✅ Standard Text-to-SQL Evaluation Results ({len(gold_only)} samples):")
    print(f"   Exact Match Accuracy: {em_acc * 100:.2f}%")
    print(f"   BLEU Score:           {bleu_score:.4f}")
    print(f"   CodeBERTScore:        {bert_score:.4f}")

# 2. DBT Task SQL Generation Evaluation (Structural & Semantic)
if valid_instruct_pairs:
    instruct_only = [pair[0] for pair in valid_instruct_pairs]
    pred_only_instruct = [pair[1] for pair in valid_instruct_pairs]

    valid_count = sum(1 for sql in pred_only_instruct if sql and any(
        kw in sql.upper() for kw in ['SELECT', 'CREATE', 'INSERT', 'UPDATE', 'DELETE', 'WITH']
    ))
    validity_rate = valid_count / len(pred_only_instruct)

    avg_sql_length = sum(len(sql) for sql in pred_only_instruct if sql) / len(pred_only_instruct)

    semantic_score = EvaluationMetrics.compute_bertscore(
        references=instruct_only,
        predictions=pred_only_instruct,
        device="cpu"
    )

    metrics_summary['SQL_Validity_Rate'] = round(validity_rate, 4)
    metrics_summary['SQL_Semantic_Score'] = round(semantic_score, 4)
    metrics_summary['SQL_Avg_Length'] = round(avg_sql_length, 1)

    print(f"\n✅ DBT Task SQL Generation Results ({len(instruct_only)} samples):")
    print(f"   Structural Validity Rate: {validity_rate * 100:.2f}% ({valid_count}/{len(pred_only_instruct)})")
    print(f"   Average SQL Length:       {avg_sql_length:.1f} characters")
    print(f"   Semantic Similarity:      {semantic_score:.4f}")

# Add all computed metrics to eval_results for visualization
if metrics_summary:
    for key, value in metrics_summary.items():
        eval_results["LoRA_Model"][key] = value
else:
    print("⚠️ No valid SQL/DBT samples were successfully processed.")

# Plot metrics (Visualizer will automatically split normalized scores from other metrics)
visualizer = ResultVisualizer(output_dir=config.outputs.plots_dir)
visualizer.plot_comparison(eval_results, save_name="model_comparison_checkpoint2.png")

print("\n✅ pipeline execution completed successfully!")